In [3]:
import requests
from datetime import datetime
import pandas as pd
import time

BASE = "https://api.openf1.org/v1"

In [4]:
def get_races(year: int):
    sessions = requests.get(f"{BASE}/sessions", params = {"year": year}).json()

    return sessions

In [5]:
# Sessions Schema
years = [2023, 2024, 2025] # 2023 is the earliest available in the API
sessions = []

for year in years:
    sessions.extend(get_races(year))

# Construct Sessions Table
session_df = pd.DataFrame(sessions)

# drop unncessary columns: meeting_key, location, country_key, country_code, country_name, circuit_short_name, gmt_offset
session_df = session_df.drop(columns=["meeting_key", "location", "country_key", "country_code", "country_name", "circuit_short_name", "gmt_offset"])

# reorder columns to match schema: session_key, session_type, session_name, circuit_key, date_start, date_end, year
session_df = session_df[["session_key", "session_type", "session_name", "circuit_key", "date_start", "date_end", "year"]]

# check for missing values
print(session_df.isnull().sum())

# save to csv
session_df.to_csv("../../data/cleaned/sessions.csv", index=False)

session_key     0
session_type    0
session_name    0
circuit_key     0
date_start      0
date_end        0
year            0
dtype: int64


In [20]:
def get_race_result(session_key: int, position: int):
    # sample curl "https://api.openf1.org/v1/session_result?session_key=7782&position%3C=3"
    # sample url: https://api.openf1.org/v1/session_result?session_key=7782&position<=3
    result = requests.get(f"{BASE}/session_result", params = {"session_key": session_key, "position<": position}).json()

    return result

In [21]:
# Race Result Schema
race_results = []
position = 20

for session_key in session_df["session_key"]:
    if session_df.loc[session_df["session_key"] == session_key, "session_type"].values[0] == "Race":
        race_results.extend(get_race_result(session_key, position = position))

        # add sleep to avoid rate limiting
        time.sleep(1)

# Construct Race Result Table
race_result_df = pd.DataFrame(race_results)

# drop unnecessary columns: meeting_key, points, duration, gap_to_leader, 
race_result_df.drop(columns=["meeting_key", "points", "duration", "gap_to_leader", "number_of_laps"], inplace=True)

# go through dnf, dns and dsq, if one is true, set driver_status to the first one that is true, if none are true, set to finished
def determine_driver_status(row):
    if row.get('dnf', False):
        return "dnf"
    elif row.get('dns', False):
        return "dns"
    elif row.get('dsq', False):
        return "dsq"
    else:
        return "finished"

race_result_df['driver_status'] = race_result_df.apply(determine_driver_status, axis=1)

# we no longer need the dnf, dns and dsq columns
race_result_df.drop(columns=["dnf", "dns", "dsq"], inplace=True)

# reorder columns to match schema: session_key, driver_number, position, driver_status
race_result_df = race_result_df[["session_key", "driver_number", "position", "driver_status"]]

# check for missing values
print(race_result_df.isnull().sum())

session_key      0
driver_number    0
position         0
driver_status    0
dtype: int64


In [ ]:
def get_overtakes(session_key: int, driver_num: int):
    # sample curl "https://api.openf1.org/v1/stints?session_key=9165&tyre_age_at_start>=3"
    # sample url: https://api.openf1.org/v1/stints?session_key=9165&tyre_age_at_start>=3 
    overtakes = requests.get(f"{BASE}/overtakes", params = {"session_key": session_key, "overtaking_driver_number": driver_num}).json()
    return overtakes

In [ ]:
# Overtakes Schema
overtakes = []

for session_key in session_df["session_key"]:
    if session_df.loc[session_df["session_key"] == session_key, "session_type"].values[0] == "Race":
        drivers_in_session = race_result_df[race_result_df["session_key"] == session_key]["driver_number"].unique()

        for driver_num in drivers_in_session:
            overtakes_data = get_overtakes(session_key=session_key, driver_num=driver_num)
            print(overtakes_data)
            overtakes.extend(overtakes_data)

        # add sleep to avoid rate limiting
        time.sleep(1)

# Overtakes Table
overtakes_df = pd.DataFrame(overtakes)


[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]


meeting_key          0
session_key          0
stint_number         0
driver_number        0
lap_start            9
lap_end              9
compound             0
tyre_age_at_start    0
dtype: int64


In [ ]:
# Laps Schema
race_sessions = session_df.loc[session_df["session_type"] == "Race", "session_key"].tolist()

mid = len(race_sessions) // 2

first_batch_sessions = race_sessions[:mid]
second_batch_sessions = race_sessions[mid:]

print("First batch:", len(first_batch_sessions), "sessions")
print("Second batch:", len(second_batch_sessions), "sessions")

laps_1 = []

for session_key in first_batch_sessions:
    # only collect laps for race sessions
    if session_df.loc[session_df["session_key"] == session_key, "session_type"].values[0] == "Race":
        drivers_in_session = race_result_df[race_result_df["session_key"] == session_key]["driver_number"].unique()
        print(drivers_in_session)

        for driver_num in drivers_in_session:
            lap_number = 1
            while True:
                lap_data = get_laps(session_key=session_key, driver_num=driver_num, lap_number=lap_number)
                print(f"{driver_num}: {lap_data}")

                if len(lap_data) == 0:
                    break

                laps_1.extend(lap_data)
                lap_number += 1

            time.sleep(1)

# Construct Laps Table
laps_df_1 = pd.DataFrame(laps_1)

# check for missing values
print(laps_df_1.isnull().sum())

# save to csv
laps_df_1.to_csv("../../data/cleaned/laps_part1.csv", index=False)



In [ ]:
# Laps Schema
race_sessions = session_df.loc[session_df["session_type"] == "Race", "session_key"].tolist()

mid = len(race_sessions) // 2

first_batch_sessions = race_sessions[:mid]
second_batch_sessions = race_sessions[mid:]

print("First batch:", len(first_batch_sessions), "sessions")
print("Second batch:", len(second_batch_sessions), "sessions")

laps_2 = []

for session_key in second_batch_sessions:
    # only collect laps for race sessions
    if session_df.loc[session_df["session_key"] == session_key, "session_type"].values[0] == "Race":
        drivers_in_session = race_result_df[race_result_df["session_key"] == session_key]["driver_number"].unique()

        for driver_num in drivers_in_session:
            lap_number = 1
            while True:
                lap_data = get_laps(session_key=session_key, driver_num=driver_num, lap_number=lap_number)
                print(lap_data)

                if len(lap_data) == 0:
                    break

                laps_2.extend(lap_data)
                lap_number += 1

                time.sleep(1)

# Construct Laps Table
laps_df_2 = pd.DataFrame(laps_2)

# check for missing values
print(laps_df_2.isnull().sum())

# save to csv
laps_df_2.to_csv("../../data/cleaned/laps_part2.csv", index=False)

laps_df_1 = pd.read_csv("../../data/cleaned/laps_part1.csv")
laps_df_2 = pd.read_csv("../../data/cleaned/laps_part2.csv")

laps_df = pd.concat([laps_df_1, laps_df_2], ignore_index=True)

laps_df.to_csv("../../data/cleaned/laps.csv", index=False)


NameError: name 'session_df' is not defined